# Sentiment multimodal FOMC — Rapport d'analyse

Ce notebook exécute le pipeline distribué de sentiment sur les événements de `data/events.json`,
récupère les scores par canal (NLP, audio, vidéo) et le signal fusionné depuis la *gateway*,
puis les compare à la variation contemporaine du S&P 500.

> **Avertissement de périmètre :** l'échantillon est petit (3 à 5 événements). Toute corrélation
> observée est **purement illustrative et non significative statistiquement**. Il faudrait au moins
> ~30 événements pour une inférence même préliminaire. Les résultats sont une preuve de concept du
> pipeline multimodal, pas une conclusion de recherche.

**Prérequis :** les services tournent (`python run_all.py`) et `python data/prepare_data.py` a été exécuté.

**Sorties :** tous les résultats sont écrits dans le dossier `results/` (cf. dernière cellule).

In [ ]:
import json
import httpx
import pandas as pd
from pathlib import Path

# Interroge la gateway pour chaque événement et joint la variation du marché.
events = json.loads(Path("data/events.json").read_text())
rows = []
for ev in events:
    res = httpx.post("http://localhost:8000/analyze", json={"event_id": ev["id"]}, timeout=600).json()
    by = {c["channel"]: c["score"] for c in res["channels"]}
    signal = json.loads(Path(f"data/processed/{ev['id']}/market_signal.json").read_text())["signal_pct"]
    rows.append({
        "event": ev["id"],
        "nlp": by.get("nlp"),
        "audio": by.get("audio"),
        "vision": by.get("vision"),
        "fusion": res["combined_score"],
        "market": signal,
    })
df = pd.DataFrame(rows)
df

In [ ]:
# Table de corrélation (logique partagée avec les tests, cf. analysis.py)
from analysis import correlation_table, plot_correlations, save_results

correlation_table(df)

In [ ]:
# Écrit TOUS les résultats dans results/ : scores.csv, correlations.csv, correlations.png
results_dir = save_results(df)
print("Résultats écrits dans :", results_dir)
for f in sorted(results_dir.glob("*")):
    print(" -", f.name)

from IPython.display import Image
Image(str(results_dir / "correlations.png"))

## Interprétation

Les graphiques et la table ci-dessus montrent la relation entre chaque canal de sentiment et la
variation du S&P 500 le jour de la conférence de presse du FOMC.

- **NLP (FinBERT)** — ton textuel de la transcription. Une corrélation positive suggérerait que le
  marché réagit favorablement à un discours accommodant (*dovish*).
- **Audio (Wav2Vec2)** — valence émotionnelle de la voix de Powell. La prosodie peut porter une
  information au-delà des mots.
- **Vidéo (DeepFace)** — affect facial. Canal *best-effort* : il peut échouer (`ok: false`) quand la
  détection de visage n'est pas fiable, et il est alors exclu de la fusion.
- **Fusion** — combinaison pondérée (NLP 50 %, audio 30 %, vidéo 20 %, renormalisée si un canal
  échoue). Censée être plus stable que chaque canal isolé.

> **Avertissement (rappel) :** avec seulement 3 à 5 points, aucune corrélation n'est significative.
> Pour conclure quoi que ce soit, étendre `data/events.json` à 30+ événements FOMC puis relancer
> `python data/prepare_data.py`.